# 🍽️ MoodMeal — Emotion-Aware Food Chatbot (Hybrid AI)
### "Feel. Eat. Heal." 🧡

Welcome to **MoodMeal**, a hybrid AI chatbot that recommends comforting foods and mindfulness tips based on your emotions.  
This notebook demonstrates the step-by-step logic used in building MoodMeal — from **emotion detection** to **food recommendation and visualization**.


In [1]:
#!pip install transformers torch nltk --quiet

### ⚙️ Step 1: Environment Setup
We start by importing the necessary libraries and checking for GPU/CPU availability.

In [2]:
# 🧩 Step 1: Setup and Configuration

import torch  # Uncommented this import to make torch available
# from transformers import pipeline
import requests
from PIL import Image
from io import BytesIO
from nltk.sentiment.vader import SentimentIntensityAnalyzer
#import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device set to use {device}")

✅ Device set to use cpu


In [3]:
import torch
import requests
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
import re, random
from PIL import Image
from io import BytesIO
import streamlit as st


In [4]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Srikar\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Srikar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [5]:
import re, random, torch
from nltk.corpus import stopwords
from transformers import pipeline


In [ ]:
from nltk.corpus import stopwords

EN_STOPWORDS = set(stopwords.words("english"))

def preprocess_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    tokens = [w for w in text.split() if w not in EN_STOPWORDS]
    return " ".join(tokens)

### 🧠 Step 2: Load Emotion and Sentiment Models
We use a Hugging Face transformer for emotion detection and NLTK's VADER for sentiment polarity scoring.


In [ ]:
# Load BERT emotion classifier
from transformers import pipeline

@st.cache_resource
def load_models():
    return pipeline(
        "text-classification",
        model="bhadresh-savani/distilbert-base-uncased-emotion",
        top_k=None
    )

emotion_classifier = load_models()

# 💭 Step 3: Emotion Detection Logic (Hybrid)

NEGATION_WORDS = ["not", "didn't", "never", "bad", "worse"]
POSITIVE_CUES = ["happy", "joy", "love", "great", "excited"]

def preprocess_and_adjust_emotion(text):
    text = text.lower()
    if any(n in text for n in NEGATION_WORDS) and any(p in text for p in POSITIVE_CUES):
        return "sadness"
    return None

    
def detect_emotion(text):
    # text_p = preprocess_text(text)
    # hunger_words = ["hungry", "starving", "want food", "need food", "empty stomach"]
    # if any(w in text_p for w in hunger_words):
    #     return "hungry"
    # result = emotion_classifier(text_p)[0]
    # return result['label'].lower()

    
    
    text_p = preprocess_text(text)

    emotion_keywords = {
        "sadness": ["sad", "upset", "unhappy", "depressed", "cry", "disappointed", "broke", "exam didn’t go well", "fail"],
        "joy": ["happy", "excited", "joy", "delighted", "great", "good"],
        "anger": ["angry", "mad", "furious", "annoyed", "irritated"],
        "fear": ["afraid", "scared", "fear", "terrified", "nervous", "anxious"],
        "love": ["love", "affection", "like", "fond", "dear"],
        "surprise": ["surprised", "amazed", "shocked", "wow"],
        "hungry": ["hungry", "starving", "food", "eat"],
    }

    # Keyword override (strong cues)
    for emotion, words in emotion_keywords.items():
        if any(w in text_p for w in words):
            return emotion,None

    # Fallback to BERT
    # Get full probability distribution from model
    result = emotion_classifier(text_p, top_k=None)[0]
    scores = {r["label"]: round(r["score"], 3) for r in result}
    top = max(result, key=lambda x: x["score"])

    if top["score"] < 0.4:
        return "neutral", scores

    return top["label"].lower(), scores

In [8]:
emotion_food_map = {
    "joy": ["smoothie", "ice cream", "fruit salad"],
    "sadness": ["dark chocolate", "soup", "mac & cheese"],
    "anger": ["herbal tea", "yogurt", "salad"],
    "fear": ["cookies", "pasta"],
    "love": [ "strawberries"],
    "surprise": ["cupcake", "popcorn"],
    "neutral": ["coffee", "sandwich"],
    "hungry": [ "burger", "sandwich"]
}

emotion_wellness_map = {
    "joy": {
        "comfort_food": ["fruit salad", "ice cream", "smoothie"],
        "wellness_tip": "Take a moment to celebrate your joy! A quick gratitude journal or sharing with a friend boosts happiness."
    },
    "sadness": {
        "comfort_food": ["dark chocolate", "soup", "mac cheese"],
        "wellness_tip": "Try 2 minutes of deep breathing and gentle stretching. Self-kindness is key during tough days."
    },
    "anger": {
        "comfort_food": ["herbal tea", "yogurt", "salad"],
        "wellness_tip": "Take a slow walk or do 5 deep breaths. Calming music may help too."
    },
    "fear": {
        "comfort_food": ["cookies", "pasta"],
        "wellness_tip": "Practice grounding: notice 5 things you see, 4 you feel, 3 you hear. Eat slowly and mindfully."
    },
    "love": {
        "comfort_food": ["strawberries", "cupcake"],
        "wellness_tip": "Call or text someone you care about. Share some food together for extra warmth."
    },
    "surprise": {
        "comfort_food": ["cupcake", "popcorn"],
        "wellness_tip": "Enjoy the surprise. Try journaling or doodling your mood."
    },
    "neutral": {
        "comfort_food": ["coffee", "sandwich"],
        "wellness_tip": "Check in on your energy level. Hydrate and stretch a bit!"
    },
    "hungry": {
        "comfort_food": ["burger", "sandwich"],
        "wellness_tip": "Eat mindfully. Sip water before meals for best digestion."
    }
}

def get_wellness_response(emotion, food):
    tips = emotion_wellness_map.get(emotion, {})
    # Pick a comfort food (or use already recommended)
    comfort = tips.get("comfort_food", [food])
    tip = tips.get("wellness_tip", "Take care of yourself!")
    food_suggestion = f"How about {random.choice(comfort)}?"
    return f"{food_suggestion}\n\n🧘 Wellness Tip: {tip}"



In [ ]:
# Load GPT-2 model for empathetic responses
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer_gpt = AutoTokenizer.from_pretrained("gpt2")
model_gpt = AutoModelForCausalLM.from_pretrained("gpt2")

def generate_empathetic_response(emotion, food):
    prompt = (f"The user feels {emotion}. "
              f"Write one empathic and friendly sentence that mentions the word '{food}' "
              f"and encourages the user positively about their feeling.")
    inputs = tokenizer_gpt.encode(prompt, return_tensors='pt')
    attention_mask = torch.ones(inputs.shape, dtype=torch.long)
    reply_ids = model_gpt.generate(
    inputs,
    attention_mask=attention_mask,
    max_length=50,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.5,
    pad_token_id=tokenizer_gpt.eos_token_id
    )
    reply = tokenizer_gpt.decode(reply_ids[0], skip_special_tokens=True)
    return reply[len(prompt):].strip()


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

c:\Users\Srikar\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Srikar\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
emotion_templates = {
    "joy": "I'm so glad you're feeling joyful! Treat yourself with some {food} to keep that energy going. 🌞",
    "sadness": "I'm sorry you’re feeling low. A little {food} might comfort you — sending a warm hug. 💙",
    "anger": "Take a deep breath. Maybe some {food} will calm your mood and refresh your mind. 🍵",
    "fear": "It’s okay to feel anxious sometimes. How about some {food} to soothe your nerves? 🌿",
    "love": "Love is in the air! Celebrate with {food} — you deserve it. ❤️",
    "surprise": "Surprises make life exciting! Pair that feeling with {food}. 🎉",
    "neutral": "Keeping it chill? A bit of {food} might make your day brighter. ☕",
    "hungry": "Your stomach’s calling! Go grab some {food} right now — you’ve earned it! 🍕"
}

def template_response(emotion, food):
    template = emotion_templates.get(emotion, "I sense you're feeling {emotion}. Maybe some {food} will help.")
    return template.format(emotion=emotion, food=food)

def safe_response(emotion, food):
    try:
        resp = generate_empathetic_response(emotion, food)
        resp_lower = resp.lower()
        if not any(k in resp_lower for k in [emotion.split()[0], food.split()[0]]):
            return template_response(emotion, food)
        return resp
    except Exception:
        return template_response(emotion, food)

def moodmeal_chatbot():
    print("🍽️ MoodMeal — Emotion-Aware Food Chatbot (Hybrid AI)")
    while True:
        user = input("\nYou: ").strip()
        if user.lower() in ["exit", "quit", "bye"]:
            print("MoodMeal: Take care and eat well! ❤️")
            break

        emotion, scores = detect_emotion(user)
        food = recommend_food(emotion)
        response = safe_response(emotion, food)

        print("MoodMeal:", response)
        from IPython.display import Image, display

        image_url = get_food_image(food)
        if image_url:
            display(Image(url=image_url))

## Image API 

In [ ]:
import requests

import os

PEXELS_API_KEY = os.getenv("PEXELS_API_KEY")

@st.cache_data
def get_food_image(food):
    url = "https://api.pexels.com/v1/search"
    headers = {"Authorization": PEXELS_API_KEY}
    params = {"query": food, "per_page": 1}
    r = requests.get(url, headers=headers, params=params)
    if r.status_code == 200 and r.json()["photos"]:
        return r.json()["photos"][0]["src"]["medium"]
    else:
        return None


2026-04-19 19:59:44.316 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [12]:
# !pip install streamlit transformers torch nltk


In [13]:
import streamlit as st
#from moodmeal import detect_emotion, recommend_food, safe_response, get_food_image

# st.title("🍽️ MoodMeal — Emotion-Aware Food Chatbot")

st.set_page_config(page_title="MoodMeal 🍽️", page_icon="🥗", layout="centered")
st.markdown("### 🧠 *Feel. Eat. Heal.* — Your AI companion for emotional wellness through food 🌿")
st.divider()

user = st.text_input("How are you feeling today?")

if user:
    emotion, scores = detect_emotion(user)
    food = recommend_food(emotion)
    reply = safe_response(emotion, food)

    # 🧠 Display Emotion Prediction Details
    if scores:
        st.subheader("📊 Model Confidence Overview")
        top_conf = max(scores.values())
        st.markdown(f"**Detected Emotion:** `{emotion.upper()}`")
        st.markdown(f"**Confidence:** `{top_conf * 100:.2f}%`")

        st.markdown("### 🔍 Emotion Probability Breakdown")
        st.bar_chart(scores)

        st.progress(top_conf)

    # 🗣️ Main Chatbot Response
    st.markdown(f"**MoodMeal:** {reply}")

    img = get_food_image(food)
    if img:
        st.image(img, caption=f"🍽️ {food}")

2026-04-19 19:59:44.593 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-19 19:59:44.596 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-19 19:59:45.215 
  command:

    streamlit run C:\Users\Srikar\AppData\Roaming\Python\Python313\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-04-19 19:59:45.218 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-19 19:59:45.220 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-19 19:59:45.221 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-19 19:59:45.223 Thread 'MainThread': missing ScriptRunContext! This warning can 

In [14]:
#moodmeal_chatbot()

In [15]:
#pip install tf-keras

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501